In [1]:
# =================================================================
# 脚本名称: V3.9_Final_Production_Engine.py
# 核心架构: 数据拉取 + V3.9特征重构 + 双轨全量训练 + 物理落盘封装
# 状态: 永久冻结 (Architecture Frozen)
# =================================================================
import os
import json
import joblib
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 自动安装并导入依赖
try:
    import supabase
    from xgboost import XGBClassifier
except ImportError:
    os.system('pip install supabase xgboost scikit-learn -q')
    import supabase
    from xgboost import XGBClassifier
from supabase import create_client, Client

print("🚨 启动 [V3.9 All-in-One Final Production Engine] 最终引擎...\n")

# =================================================================
# 2. V3.9 官方概率收缩层定义 (必须在顶层声明，供序列化使用)
# =================================================================
class V3_9_ProbabilityLayer:
    def __init__(self, team_weight=0.6, player_weight=0.4, soft_shrink=0.8, penalty_shrink=0.7):
        self.tw = team_weight
        self.pw = player_weight
        self.ss = soft_shrink
        self.ps = penalty_shrink

    def predict(self, p_team, p_player, team_diffs, player_diffs):
        p_t, p_p = np.asarray(p_team), np.asarray(p_player)
        t_d, p_d = np.asarray(team_diffs), np.asarray(player_diffs)

        # 原生 0.6 / 0.4 融合
        raw = (p_t * self.tw) + (p_p * self.pw)
        # 0.8 线性柔性收缩
        soft = 0.5 + (raw - 0.5) * self.ss

        # 置信度交叉惩罚防爆冷
        final = soft.copy()
        mask_penalty = (np.abs(t_d) > 15.0) & (np.abs(p_d) < 3.0)
        final[mask_penalty] = 0.5 + (soft[mask_penalty] - 0.5) * self.ps
        return final

# =================================================================
# 3. 数据库连接与全量数据清洗
# =================================================================
SUPABASE_URL = "https://wfqfjrfkcejompigcqkv.supabase.co"
SUPABASE_KEY = "sb_publishable_U44mIM70vqcuAdGTOKYXdA_GFodRJRg"
db_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

def fetch_all(table, columns):
    data, start, page_size = [], 0, 1000
    while True:
        res = db_client.table(table).select(columns).range(start, start + page_size - 1).execute()
        if not res.data: break
        data.extend(res.data)
        if len(res.data) < page_size: break
        start += page_size
    return pd.DataFrame(data)

def is_regular_season(date_val):
    return (date_val.month > 5) or (date_val.month == 5 and date_val.day >= 8)

print("📦 正在从云端提取底层数据，重构宏微观特征...")
df_games = fetch_all("WNBA_Game_Features_v2", "match_id, match_date_bj, home_team_cn, away_team_cn, home_score, away_score")
df_games.rename(columns={'match_id': 'game_id', 'match_date_bj': 'game_date'}, inplace=True)
df_games['game_date'] = pd.to_datetime(df_games['game_date'])
df_games = df_games[df_games['game_date'].apply(is_regular_season)]
df_games['season'] = df_games['game_date'].dt.year.astype(str)
df_games = df_games.sort_values('game_date').dropna(subset=['home_score', 'away_score']).reset_index(drop=True)

df_box = fetch_all("WNBA_Player_Boxscore", "game_id, game_date, team_id, player_id, minutes, points, field_goal_attempts, free_throw_attempts, turnovers")
df_box['game_date'] = pd.to_datetime(df_box['game_date'])
df_box = df_box[df_box['game_date'].apply(is_regular_season)]
df_box['season'] = df_box['game_date'].dt.year.astype(str)

df_rating = fetch_all("Player_Rating", "player_id, game_date, overall_rating, efficiency_rating")
df_rating['game_date'] = pd.to_datetime(df_rating['game_date'])
df_rating = df_rating.drop_duplicates(subset=['player_id', 'game_date'], keep='last')

df_box['points'] = pd.to_numeric(df_box['points'], errors='coerce').fillna(0)
for col in ['field_goal_attempts', 'free_throw_attempts', 'turnovers']:
    df_box[col] = pd.to_numeric(df_box[col], errors='coerce').fillna(0)

def parse_minutes(m_str):
    try:
        parts = str(m_str).split(':')
        if len(parts) >= 2: return float(parts[0]) + float(parts[1]) / 60.0
        return float(parts[0])
    except: return 0.0
df_box['minutes_num'] = df_box['minutes'].apply(parse_minutes)
df_box['possessions'] = df_box['field_goal_attempts'] + 0.44 * df_box['free_throw_attempts'] + df_box['turnovers']

team_points = df_box.groupby(['game_id', 'team_id'])['points'].sum().reset_index()
h_tids, a_tids = [], []
for idx, row in df_games.iterrows():
    gid, h_score, a_score = str(row['game_id']), int(float(row['home_score'])), int(float(row['away_score']))
    match_teams = team_points[team_points['game_id'] == gid]
    h_id, a_id = None, None
    if len(match_teams) == 2:
        t1_id, t1_pts = match_teams.iloc[0]['team_id'], match_teams.iloc[0]['points']
        t2_id, t2_pts = match_teams.iloc[1]['team_id'], match_teams.iloc[1]['points']
        if t1_pts == h_score and t2_pts == a_score: h_id, a_id = t1_id, t2_id
        elif t2_pts == h_score and t1_pts == a_score: h_id, a_id = t2_id, t1_id
        else:
            if abs(t1_pts - h_score) + abs(t2_pts - a_score) <= abs(t2_pts - h_score) + abs(t1_pts - a_score): h_id, a_id = t1_id, t2_id
            else: h_id, a_id = t2_id, t1_id
    h_tids.append(h_id)
    a_tids.append(a_id)

df_games['home_team_id'] = h_tids
df_games['away_team_id'] = a_tids
df_games = df_games.dropna(subset=['home_team_id', 'away_team_id']).copy()

TEAM_ID_MAP = {"1611661330": "GSV_2026", "1611661331": "TOR_2026", "1611661332": "POR_2026", "1611661317_MERCURY": "1611661321"}
for col in ['home_team_id', 'away_team_id']: df_games[col] = df_games[col].astype(str).replace(TEAM_ID_MAP)
df_box['team_id'] = df_box['team_id'].astype(str).replace(TEAM_ID_MAP)

team_game_logs = []
df_games['score_diff_raw'] = df_games['home_score'] - df_games['away_score']
for idx, row in df_games.iterrows():
    h_id, a_id, date, s_diff = row['home_team_id'], row['away_team_id'], row['game_date'], row['score_diff_raw']
    team_game_logs.append({'team_id': h_id, 'game_date': date, 'net_rtg': s_diff})
    team_game_logs.append({'team_id': a_id, 'game_date': date, 'net_rtg': -s_diff})
df_team_logs = pd.DataFrame(team_game_logs).sort_values(by=['team_id', 'game_date']).reset_index(drop=True)

def get_team_macro_strength(t_id, t_date):
    past = df_team_logs[(df_team_logs['team_id'] == t_id) & (df_team_logs['game_date'] < t_date)]
    if past.empty: return 0.0, 7.0
    recent_net = past['net_rtg'].tail(10).mean()
    rest_days = min(float((t_date - past['game_date'].max()).days), 7.0)
    return recent_net, rest_days

def calc_v3_7_player_impact(team_id, t_date, t_gid, t_season):
    past_box = df_box[(df_box['team_id'] == team_id) & (df_box['game_date'] < t_date) & (df_box['season'] == t_season)]
    roster_box = df_box[(df_box['game_id'] == t_gid) & (df_box['team_id'] == team_id)]

    if past_box.empty:
        if roster_box.empty: return 5.0
        active_pids = roster_box['player_id'].unique()
        historical_top3 = list(active_pids[:3])
        roles = list(active_pids[3:])
        player_stats = pd.DataFrame({'player_id': active_pids, 'avg_minutes': 20.0, 'avg_usg': 10.0, 'avg_ts': 0.53})
    else:
        recent_10_dates = past_box['game_date'].drop_duplicates().nlargest(10)
        recent_box = past_box[past_box['game_date'].isin(recent_10_dates)]
        player_stats = recent_box.groupby('player_id').agg(
            avg_minutes=('minutes_num', 'mean'), avg_usg=('possessions', 'mean'),
            tot_pts=('points', 'sum'), tot_fga=('field_goal_attempts', 'sum'), tot_fta=('free_throw_attempts', 'sum')
        ).reset_index()

        player_stats['avg_ts'] = player_stats['tot_pts'] / (2 * (player_stats['tot_fga'] + 0.44 * player_stats['tot_fta']) + 1e-5)
        player_stats['avg_ts'] = player_stats['avg_ts'].clip(0.30, 0.75)
        player_stats = player_stats.sort_values(by='avg_minutes', ascending=False)
        historical_top3 = player_stats.head(3)['player_id'].tolist()
        roles = player_stats.iloc[3:8]['player_id'].tolist()

    active_pids = roster_box['player_id'].tolist() if not roster_box.empty else historical_top3 + roles
    missing_core_count = sum(1 for pid in historical_top3 if pid not in active_pids)
    absence_penalty = missing_core_count * 4.0

    def get_player_composite_score(pid):
        pr = df_rating[(df_rating['player_id'] == pid) & (df_rating['game_date'] < t_date)]
        if pr.empty: base_rtg, base_ts = 3.0, 0.50
        else:
            latest = pr.sort_values('game_date', ascending=False).iloc[0]
            base_rtg = latest['overall_rating']
            base_ts = latest['efficiency_rating'] / 100.0 if latest['efficiency_rating'] > 0 else 0.50

        p_row = player_stats[player_stats['player_id'] == pid]
        minutes = p_row['avg_minutes'].values[0] if not p_row.empty else 15.0
        usg = p_row['avg_usg'].values[0] if not p_row.empty else 5.0
        ts = p_row['avg_ts'].values[0] if not p_row.empty else base_ts

        form = 0.0
        if not pr.empty and len(pr) >= 5:
            sorted_pr = pr.sort_values('game_date', ascending=False)
            recent_mean = sorted_pr.head(5)['overall_rating'].mean()
            hist_mean = sorted_pr.tail(15)['overall_rating'].mean() if len(pr) > 15 else sorted_pr['overall_rating'].mean()
            form = recent_mean - hist_mean

        return (base_rtg * 2.5) + (form * 3.5) + (ts * 12.0) + (usg * 0.4) + (minutes * 0.15)

    top1_val = get_player_composite_score(historical_top3[0]) if len(historical_top3) >= 1 else 10.0
    top2_val = get_player_composite_score(historical_top3[1]) if len(historical_top3) >= 2 else 8.0
    top3_val = get_player_composite_score(historical_top3[2]) if len(historical_top3) >= 3 else 6.0
    role_scores = [get_player_composite_score(pid) for pid in roles]
    role_val = sum(role_scores) / len(role_scores) if role_scores else 5.0

    player_impact = (top1_val * 0.40) + (top2_val * 0.30) + (top3_val * 0.20) + (role_val * 0.10)
    return player_impact - absence_penalty

features_list = []
for idx, row in df_games.iterrows():
    t_date, t_season, t_gid = row['game_date'], str(row['season']), str(row['game_id'])
    h_id, a_id = row['home_team_id'], row['away_team_id']

    h_macro, h_rest = get_team_macro_strength(h_id, t_date)
    a_macro, a_rest = get_team_macro_strength(a_id, t_date)
    team_strength_diff = h_macro - a_macro
    clamped_team_diff = np.clip(team_strength_diff, -20.0, 20.0)

    h_player_impact = calc_v3_7_player_impact(h_id, t_date, t_gid, t_season)
    a_player_impact = calc_v3_7_player_impact(a_id, t_date, t_gid, t_season)
    player_impact_diff = h_player_impact - a_player_impact

    rest_days_diff = h_rest - a_rest
    fatigue_diff = (1.0 if h_rest <= 1 else 0.0) - (1.0 if a_rest <= 1 else 0.0)

    h_score, a_score = int(row['home_score']), int(row['away_score'])
    features_list.append({
        'game_id': t_gid, 'game_date': t_date, 'season': t_season,
        'home_team': row['home_team_cn'], 'away_team': row['away_team_cn'],
        'team_strength_diff': round(clamped_team_diff, 3),
        'player_impact_diff': round(player_impact_diff, 3),
        'home_advantage': 1.0,
        'rest_days_diff': round(rest_days_diff, 1),
        'fatigue_diff': round(fatigue_diff, 1),
        'is_home_win': 1 if h_score > a_score else 0
    })

full_train_df = pd.DataFrame(features_list)

# =================================================================
# 4. 全量训练模型并执行物理落盘
# =================================================================
print("⚙️ 特征合成完毕！正在执行 V3.9 极净全量训练与落盘...")
X_team_cols = ['team_strength_diff', 'home_advantage', 'rest_days_diff', 'fatigue_diff']
X_player_cols = ['player_impact_diff']
y_full = full_train_df['is_home_win']

xgb_params = {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42}
model_team_prod = XGBClassifier(**xgb_params).fit(full_train_df[X_team_cols], y_full)
model_player_prod = XGBClassifier(**xgb_params).fit(full_train_df[X_player_cols], y_full)
prob_layer_prod = V3_9_ProbabilityLayer()

os.makedirs("models", exist_ok=True)
joblib.dump({"team_node": model_team_prod, "player_node": model_player_prod}, "models/v3_9_fusion_model.pkl")
joblib.dump(prob_layer_prod, "models/v3_9_probability_layer.pkl")

feature_schema = {
    "version": "V3.9 Production Candidate",
    "team_features": X_team_cols, "player_features": X_player_cols,
    "fusion_weights": {"team_prob": 0.6, "player_prob": 0.4},
    "adjustments": {"soft_shrink_factor": 0.8, "penalty_shrink_factor": 0.7}
}
with open("models/feature_schema.json", "w", encoding="utf-8") as f:
    json.dump(feature_schema, f, indent=4, ensure_ascii=False)

# =================================================================
# 5. 生成实盘对照表 (以近期10场为例)
# =================================================================
print("📊 正在输出标准 V3.9 实盘预测宽表...")
recent_df = full_train_df.tail(10).copy()
p_t = model_team_prod.predict_proba(recent_df[X_team_cols])[:, 1]
p_p = model_player_prod.predict_proba(recent_df[X_player_cols])[:, 1]
p_final = prob_layer_prod.predict(p_t, p_p, recent_df['team_strength_diff'], recent_df['player_impact_diff'])

recent_df['prediction'] = np.where(p_final >= 0.5, 1, 0)
recent_df['prediction_side'] = np.where(recent_df['prediction'] == 1, "HOME", "AWAY")
recent_df['final_probability'] = np.round(p_final, 4)

output_view = recent_df[['game_date', 'home_team', 'away_team', 'team_strength_diff', 'player_impact_diff', 'final_probability', 'prediction_side']]
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n" + "="*80)
print(output_view.to_string(index=False))
print("="*80)
print("\n✅ 老大，引擎文件已全量保存在当前环境的 models/ 文件夹下，代码主线干净无污染！这是您唯一的 V3.9_Final！")

🚨 启动 [V3.9 All-in-One Final Production Engine] 最终引擎...

📦 正在从云端提取底层数据，重构宏微观特征...
⚙️ 特征合成完毕！正在执行 V3.9 极净全量训练与落盘...
📊 正在输出标准 V3.9 实盘预测宽表...

 game_date            home_team                away_team  team_strength_diff  player_impact_diff  final_probability prediction_side
2026-07-11    Connecticut Sun W Golden State Valkyries W                -5.9              -3.054             0.4397            AWAY
2026-07-11      Toronto Tempo W           Dallas Wings W                -7.4              -5.935             0.3604            AWAY
2026-07-11 Los Angeles Sparks W            Chicago Sky W                -6.2              10.722             0.5449            HOME
2026-07-12     Las Vegas Aces W        Phoenix Mercury W                 0.6              13.517             0.5632            HOME
2026-07-12      Atlanta Dream W          Portland Fire W                14.5              -3.317             0.5455            HOME
2026-07-12     Minnesota Lynx W       New York Liberty W             